# Type-1 Exponential DM/EDE Coupling With `classy`

This notebook runs the new type-1 exponential mass coupling,

`m_DM(phi) = m0 exp(c_idm_ede phi)`,

using the local `classy` wrapper. It writes the CLASS background output and prints the basic EDE timing diagnostics.

In [ ]:
from pathlib import Path
import ctypes
import math
import os
import sys

repo = Path.cwd()
if not (repo / 'class').exists():
    repo = Path('/Users/vpoulin/Dropbox/Labo/ProgrammeCMB/AxiCLASS_EDE_DM_coupling')

def preload_openmp_runtime():
    candidates = [
        os.environ.get('CLASSY_GOMP_LIB'),
        '/usr/local/Cellar/gcc@12/12.3.0/lib/gcc/12/libgomp.1.dylib',
        '/opt/homebrew/opt/gcc/lib/gcc/current/libgomp.1.dylib',
        '/usr/local/opt/gcc/lib/gcc/current/libgomp.1.dylib',
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            ctypes.CDLL(candidate, mode=ctypes.RTLD_GLOBAL)
            return candidate
    return None

preload_openmp_runtime()
sys.path.insert(0, str(repo))
from classy import Class


In [ ]:
root = repo / 'output' / 'example_type1_exp_classy_notebook_'

params = {
    'root': str(root),
    'output': 'tCl',
    'write background': 'yes',
    'write parameters': 'no',
    'omega_b': 0.02251,
    'omega_ini_idm_ede': 0.12,
    'H0': 72.81,
    'tau_reio': 0.068,
    'A_s': 2.191e-9,
    'n_s': 0.9860,
    'coupling_type': 1,
    'idm_ede_mass_form': 'exp',
    'c_idm_ede': 0.1,
    'N_ur': 2.0328,
    'N_ncdm': 1,
    'deg_ncdm': 1,
    'm_ncdm': 0.06,
    'T_ncdm': 0.71611,
    'scf_potential': 'axion',
    'n_axion': 3,
    'scf_parameters': '3.141592,0.0',
    'f_axion': 0.3,
    'm_axion': 300.0,
    'scf_evolve_as_fluid': 'no',
    'scf_evolve_like_axionCAMB': 'no',
    'do_shooting': 'yes',
    'do_shooting_scf': 'no',
    'scf_has_perturbations': 'no',
    'attractor_ic_scf': 'no',
    'modes': 's',
    'gauge': 'synchronous',
    'lensing': 'no',
}

cosmo = Class()
cosmo.set(**params)
cosmo.compute()
cosmo.struct_cleanup()
cosmo.empty()


In [ ]:
def read_background(path):
    rows = []
    with open(path) as handle:
        for line in handle:
            if line.startswith('#') or not line.strip():
                continue
            rows.append([float(x) for x in line.split()])
    return rows

def nearest_row(rows, z_target):
    target = math.log1p(z_target)
    return min(rows, key=lambda row: abs(math.log1p(row[0]) - target))

def phi_over_hubble(row):
    a = 1.0 / (1.0 + row[0])
    return row[24] / (a * row[3])

bg_file = sorted(root.parent.glob(root.name + '*background.dat'), key=lambda path: path.stat().st_mtime)[-1]
rows = read_background(bg_file)
eq = nearest_row(rows, 3400.0)
peak = max(rows, key=lambda row: row[18])

print('background =', bg_file)
print('z_peak(fEDE) =', peak[0])
print('fEDE_peak =', peak[18])
print('fEDE(z_eq ~= 3400) =', eq[18])
print("phi_prime/(aH) at z_eq =", phi_over_hubble(eq))
print('Veff_phi/(H^2 phi) at z_eq =', eq[29])
